In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Step 1: Read API configuration CSV

config_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("Files/api_ingestion_config.csv")

# Display configuration
display(config_df)




StatementMeta(, f5a07e03-7150-465d-ade6-4b67bd4d7dd7, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ecebb419-5a16-472a-919c-7e3e8f309446)

Metadata table created successfully!


In [3]:
# Create a metadata configuration table in Lakehouse

config_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("config_api_endpoints")

print("Metadata table created successfully!")

StatementMeta(, f5a07e03-7150-465d-ade6-4b67bd4d7dd7, 5, Finished, Available, Finished, False)

Metadata table created successfully!


In [4]:
# Read the newly created metadata table

metadata_df = spark.read.table("config_api_endpoints")

display(metadata_df)

StatementMeta(, f5a07e03-7150-465d-ade6-4b67bd4d7dd7, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3357aee8-1684-428d-8f33-65b575db9fb2)

In [5]:
# Read active API configurations

active_apis = spark.sql("""
    SELECT *
    FROM dbo.config_api_endpoints
    WHERE is_active = 1
""")

display(active_apis)

StatementMeta(, f5a07e03-7150-465d-ade6-4b67bd4d7dd7, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 01905fc7-a941-452e-8ef6-55349c6bdbd2)

In [6]:
# Display API endpoints that need ingestion

active_apis.select(
    "tenant_id",
    "api_name",
    "endpoint",
    "target_table"
).show(truncate=False)

StatementMeta(, f5a07e03-7150-465d-ade6-4b67bd4d7dd7, 8, Finished, Available, Finished, False)

+---------+------------+---------+---------------+
|tenant_id|api_name    |endpoint |target_table   |
+---------+------------+---------+---------------+
|T001     |Users API   |/users   |bronze_users   |
|T001     |Products API|/products|bronze_products|
|T002     |Carts API   |/carts   |bronze_carts   |
|T002     |Posts API   |/posts   |bronze_posts   |
+---------+------------+---------+---------------+



In [1]:
#This checks how many records were actually loaded into each Bronze table.
tables = [
    "bronze_users",
    "bronze_products",
    "bronze_carts",
    "bronze_posts"
]

for table in tables:
    count = spark.sql(
        f"SELECT COUNT(*) AS total FROM dbo.{table}"
    ).collect()[0]["total"]

    print(f"{table}: {count} records")

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 3, Finished, Available, Finished, False)

bronze_users: 30 records
bronze_products: 273 records
bronze_carts: 114 records
bronze_posts: 82 records


In [2]:

tables = [
    "bronze_users",
    "bronze_products",
    "bronze_carts",
    "bronze_posts"
]

for table in tables:
    print("\n" + "=" * 60)
    print(f"TABLE: {table}")
    print("=" * 60)

    df = spark.table(f"dbo.{table}")

    print("SCHEMA:")
    df.printSchema()

    print("SAMPLE RECORDS:")
    display(df.limit(5))


StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 4, Finished, Available, Finished, False)


TABLE: bronze_users
SCHEMA:
root
 |-- users.id: long (nullable = true)
 |-- users.firstName: string (nullable = true)
 |-- users.lastName: string (nullable = true)
 |-- users.maidenName: string (nullable = true)
 |-- users.age: long (nullable = true)
 |-- users.gender: string (nullable = true)
 |-- users.email: string (nullable = true)
 |-- users.phone: string (nullable = true)
 |-- users.username: string (nullable = true)
 |-- users.password: string (nullable = true)
 |-- users.birthDate: string (nullable = true)
 |-- users.image: string (nullable = true)
 |-- users.bloodGroup: string (nullable = true)
 |-- users.height: double (nullable = true)
 |-- users.weight: double (nullable = true)
 |-- users.eyeColor: string (nullable = true)
 |-- users.hair.color: string (nullable = true)
 |-- users.hair.type: string (nullable = true)
 |-- users.ip: string (nullable = true)
 |-- users.address.address: string (nullable = true)
 |-- users.address.city: string (nullable = true)
 |-- users.addre

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `users`.`id` cannot be resolved. Did you mean one of the following? [`users`.`id`, `users`.`firstName`, `users`.`lastName`, `users`.`maidenName`, `users`.`age`, `users`.`gender`, `users`.`email`, `users`.`phone`, `users`.`username`, `users`.`password`, `users`.`birthDate`, `users`.`image`, `users`.`bloodGroup`, `users`.`height`, `users`.`weight`, `users`.`eyeColor`, `users`.`hair`.`color`, `users`.`hair`.`type`, `users`.`ip`, `users`.`address`.`address`, `users`.`address`.`city`, `users`.`address`.`state`, `users`.`address`.`stateCode`, `users`.`address`.`postalCode`, `users`.`address`.`coordinates`.`lat`, `users`.`address`.`coordinates`.`lng`, `users`.`address`.`country`, `users`.`macAddress`, `users`.`university`, `users`.`bank`.`cardExpire`, `users`.`bank`.`cardNumber`, `users`.`bank`.`cardType`, `users`.`bank`.`currency`, `users`.`bank`.`iban`, `users`.`company`.`department`, 

In [4]:
# Data profiling 


from pyspark.sql import functions as F

tables = [
    "bronze_users",
    "bronze_products",
    "bronze_carts",
    "bronze_posts"
]

for table in tables:
    df = spark.table(f"dbo.{table}")

    print("\n" + "=" * 60)
    print(f"DATA PROFILE: {table}")
    print("=" * 60)

    print("Total rows:", df.count())
    print("Total columns:", len(df.columns))

    # Count NULL values in every column
    null_counts = df.select([
        F.count(
            F.when(F.col(f"`{c}`").isNull(), 1)
        ).alias(f"null_{i}")
        for i, c in enumerate(df.columns)
    ])

    print("NULL VALUES PER COLUMN:")
    display(null_counts)

    # Count duplicate rows
    print("Duplicate rows:", df.count() - df.dropDuplicates().count())



StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 6, Finished, Available, Finished, False)


DATA PROFILE: bronze_users
Total rows: 30
Total columns: 55
NULL VALUES PER COLUMN:


SynapseWidget(Synapse.DataFrame, 3ed3356a-8a1a-439f-afa4-5d5f23364a3c)

Duplicate rows: 0

DATA PROFILE: bronze_products
Total rows: 273
Total columns: 34
NULL VALUES PER COLUMN:


SynapseWidget(Synapse.DataFrame, 86154f83-3f42-456c-87b2-e3da1f8cd35e)

Duplicate rows: 0

DATA PROFILE: bronze_carts
Total rows: 114
Total columns: 17
NULL VALUES PER COLUMN:


SynapseWidget(Synapse.DataFrame, c261bdd1-f43f-4343-90bf-d9f8a89f8aaf)

Duplicate rows: 1

DATA PROFILE: bronze_posts
Total rows: 82
Total columns: 11
NULL VALUES PER COLUMN:


SynapseWidget(Synapse.DataFrame, d4cc5bbc-2254-48d9-8404-14e539174c07)

Duplicate rows: 0


In [5]:

from pyspark.sql import functions as F

for table in tables:
    df = spark.table(f"dbo.{table}")

    total_rows = df.count()
    duplicate_rows = total_rows - df.dropDuplicates().count()

    null_total = sum(
        df.filter(F.col(f"`{c}`").isNull()).count()
        for c in df.columns
    )

    print(f"\nTABLE: {table}")
    print("Rows:", total_rows)
    print("Columns:", len(df.columns))
    print("Total NULL values:", null_total)
    print("Duplicate rows:", duplicate_rows)


StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 7, Finished, Available, Finished, False)


TABLE: bronze_users
Rows: 30
Columns: 55
Total NULL values: 0
Duplicate rows: 0

TABLE: bronze_products
Rows: 273
Columns: 34
Total NULL values: 63
Duplicate rows: 0

TABLE: bronze_carts
Rows: 114
Columns: 17
Total NULL values: 0
Duplicate rows: 1

TABLE: bronze_posts
Rows: 82
Columns: 11
Total NULL values: 0
Duplicate rows: 0


In [8]:
# check nulls 


from pyspark.sql import functions as F

df_products = spark.table("dbo.bronze_products")

null_profile = df_products.select([
    F.count(
        F.when(
            F.col(f"`{c}`").isNull(), 1
        )
    ).alias(f"null_col_{i}")
    for i, c in enumerate(df_products.columns)
])

display(null_profile)


StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52c5ab57-991c-44f7-a839-2e2ec22564b3)

In [9]:
df_products = spark.table("dbo.bronze_products")

print("Rows:", df_products.count())
print("Columns:", len(df_products.columns))

for c in df_products.columns:
    count = df_products.filter(
        F.col(f"`{c}`").isNull()
    ).count()

    if count > 0:
        print(c, ":", count)

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 11, Finished, Available, Finished, False)

Rows: 273
Columns: 34
products.brand : 63


In [10]:
# missing brand records
df_products.filter(
    F.col("`products.brand`").isNull()
).select(
    "`products.id`",
    "`products.title`",
    "`products.category`",
    "`products.brand`"
).show(20, truncate=False)

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 12, Finished, Available, Finished, False)

+-----------+--------------+-----------------+--------------+
|products.id|products.title|products.category|products.brand|
+-----------+--------------+-----------------+--------------+
|16         |Apple         |groceries        |NULL          |
|16         |Apple         |groceries        |NULL          |
|16         |Apple         |groceries        |NULL          |
|17         |Beef Steak    |groceries        |NULL          |
|17         |Beef Steak    |groceries        |NULL          |
|17         |Beef Steak    |groceries        |NULL          |
|18         |Cat Food      |groceries        |NULL          |
|18         |Cat Food      |groceries        |NULL          |
|18         |Cat Food      |groceries        |NULL          |
|18         |Cat Food      |groceries        |NULL          |
|18         |Cat Food      |groceries        |NULL          |
|18         |Cat Food      |groceries        |NULL          |
|19         |Chicken Meat  |groceries        |NULL          |
|19     

In [13]:
# duplicate carts

df_carts = spark.table("dbo.bronze_carts")

total_rows = df_carts.count()
unique_rows = df_carts.distinct().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)


StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 15, Finished, Available, Finished, False)

Total rows: 114
Unique rows: 113
Duplicate rows: 1


In [14]:
# create silver User tables
from pyspark.sql import functions as F

# Read Bronze Users
df_users = spark.table("dbo.bronze_users")

# Rename dotted columns and remove sensitive fields
users_clean = df_users.select([
    F.col(f"`{c}`").alias(
        c.replace("users.", "").replace(".", "_")
    )
    for c in df_users.columns
    if c not in [
        "users.password",
        "users.ssn",
        "users.bank.cardNumber",
        "users.bank.iban"
    ]
])

# Remove exact duplicate records
users_clean = users_clean.dropDuplicates()

# Save Silver table
users_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_users")

print("Silver Users created:", users_clean.count())
display(users_clean.limit(5))


StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 16, Finished, Available, Finished, False)

Silver Users created: 30


SynapseWidget(Synapse.DataFrame, 5ce79923-bdbd-41d5-9519-7389e5a386ad)

In [15]:
#Create Silver Products
df_products = spark.table("dbo.bronze_products")

products_clean = df_products.select([
    F.col(f"`{c}`").alias(
        c.replace("products.", "").replace(".", "_")
    )
    for c in df_products.columns
])

# Replace missing brand values
products_clean = products_clean.fillna(
    {"brand": "Unknown"}
)

products_clean = products_clean.dropDuplicates()

products_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_products")

print("Silver Products created:", products_clean.count())
display(products_clean.limit(5))


StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 17, Finished, Available, Finished, False)

Silver Products created: 273


SynapseWidget(Synapse.DataFrame, eca54680-fbc5-49a0-a070-9abc6fdf58eb)

In [16]:
# create silver carts 

df_carts = spark.table("dbo.bronze_carts")

carts_clean = df_carts.select([
    F.col(f"`{c}`").alias(
        c.replace("carts.", "").replace(".", "_")
    )
    for c in df_carts.columns
])

carts_clean = carts_clean.dropDuplicates()

carts_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_carts")

print("Silver Carts created:", carts_clean.count())
display(carts_clean.limit(5))


StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 18, Finished, Available, Finished, False)

AnalysisException: [COLUMN_ALREADY_EXISTS] The column `total` already exists. Choose another name or rename the existing column. SQLSTATE: 42711

In [18]:
#investigate the cart
df_carts = spark.table("dbo.bronze_carts")

print("Original columns:")
for c in df_carts.columns:
    print(c)

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 20, Finished, Available, Finished, False)

Original columns:
carts.id
carts.products.id
carts.products.title
carts.products.price
carts.products.quantity
carts.products.total
carts.products.discountPercentage
carts.products.discountedTotal
carts.products.thumbnail
carts.total
carts.discountedTotal
carts.userId
carts.totalProducts
carts.totalQuantity
total
skip
limit


In [19]:
#corrected Silver Carts code
from pyspark.sql import functions as F

df_carts = spark.table("dbo.bronze_carts")

# Remove API response metadata
exclude_cols = ["total", "skip", "limit"]

# Clean column names and remove duplicate rows
carts_clean = df_carts.select([
    F.col(f"`{c}`").alias(
        c.replace("carts.", "").replace(".", "_")
    )
    for c in df_carts.columns
    if c not in exclude_cols
])

carts_clean = carts_clean.dropDuplicates()

# Save Silver table
carts_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_carts")

print("Silver Carts created:", carts_clean.count())
display(carts_clean.limit(5))

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 21, Finished, Available, Finished, False)

Silver Carts created: 113


SynapseWidget(Synapse.DataFrame, d8b12dea-155b-4c02-9f70-6ff5a528c29e)

In [17]:
#Silver posts

df_posts = spark.table("dbo.bronze_posts")

posts_clean = df_posts.select([
    F.col(f"`{c}`").alias(
        c.replace("posts.", "").replace(".", "_")
    )
    for c in df_posts.columns
])

posts_clean = posts_clean.dropDuplicates()

posts_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_posts")

print("Silver Posts created:", posts_clean.count())
display(posts_clean.limit(5))


StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 19, Finished, Available, Finished, False)

Silver Posts created: 82


SynapseWidget(Synapse.DataFrame, 609d7719-8dc4-4054-81f5-4d25254c6eb0)

In [20]:
#verify all silver tables 
for table in [
    "silver_users",
    "silver_products",
    "silver_carts",
    "silver_posts"
]:
    df = spark.table(f"dbo.{table}")
    print(table, ":", df.count(), "records")

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 22, Finished, Available, Finished, False)

silver_users : 30 records
silver_products : 273 records
silver_carts : 113 records
silver_posts : 82 records


In [21]:
#Silver data quality validation.
from pyspark.sql import functions as F

tables = [
    "silver_users",
    "silver_products",
    "silver_carts",
    "silver_posts"
]

for table in tables:
    df = spark.table(f"dbo.{table}")

    print("\nTABLE:", table)
    print("Rows:", df.count())
    print("Columns:", len(df.columns))
    print("Duplicate rows:", df.count() - df.distinct().count())

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 23, Finished, Available, Finished, False)


TABLE: silver_users
Rows: 30
Columns: 51
Duplicate rows: 0

TABLE: silver_products
Rows: 273
Columns: 34
Duplicate rows: 0

TABLE: silver_carts
Rows: 113
Columns: 14
Duplicate rows: 0

TABLE: silver_posts
Rows: 82
Columns: 11
Duplicate rows: 0


In [22]:
#validate prim key and ID
from pyspark.sql import functions as F

checks = {
    "silver_users": ["id"],
    "silver_products": ["id"],
    "silver_carts": ["id", "userId"],
    "silver_posts": ["id", "userId"]
}

for table, required_columns in checks.items():

    df = spark.table(f"dbo.{table}")

    print(f"\n--- {table} ---")

    for col_name in required_columns:

        if col_name in df.columns:

            null_count = df.filter(
                F.col(f"`{col_name}`").isNull()
            ).count()

            print(
                f"{col_name}: NULL values = {null_count}"
            )

        else:
            print(f"{col_name}: Column not found")

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 24, Finished, Available, Finished, False)


--- silver_users ---
id: NULL values = 0

--- silver_products ---
id: NULL values = 0

--- silver_carts ---
id: NULL values = 0
userId: NULL values = 0

--- silver_posts ---
id: NULL values = 0
userId: NULL values = 0


In [23]:
tables = [
    "silver_users",
    "silver_products",
    "silver_carts",
    "silver_posts"
]

for table in tables:
    df = spark.table(f"dbo.{table}")

    print(f"\n--- {table} ---")
    df.printSchema()

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 25, Finished, Available, Finished, False)


--- silver_users ---
root
 |-- id: long (nullable = true)
 |-- firstName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- maidenName: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- username: string (nullable = true)
 |-- birthDate: string (nullable = true)
 |-- image: string (nullable = true)
 |-- bloodGroup: string (nullable = true)
 |-- height: double (nullable = true)
 |-- weight: double (nullable = true)
 |-- eyeColor: string (nullable = true)
 |-- hair_color: string (nullable = true)
 |-- hair_type: string (nullable = true)
 |-- ip: string (nullable = true)
 |-- address_address: string (nullable = true)
 |-- address_city: string (nullable = true)
 |-- address_state: string (nullable = true)
 |-- address_stateCode: string (nullable = true)
 |-- address_postalCode: string (nullable = true)
 |-- address_coordinates_lat: double (nullabl

In [24]:
# data typess
tables = {
    "silver_users": ["id", "age"],
    "silver_products": ["id", "price", "stock", "rating"],
    "silver_carts": ["id", "userId", "total", "discountedTotal"],
    "silver_posts": ["id", "userId", "views"]
}

for table, columns in tables.items():
    df = spark.table(f"dbo.{table}")

    print(f"\n{table}")
    for c in columns:
        if c in df.columns:
            print(c, ":", df.schema[c].dataType)

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 26, Finished, Available, Finished, False)


silver_users
id : LongType()
age : LongType()

silver_products
id : LongType()
price : DoubleType()
stock : LongType()
rating : DoubleType()

silver_carts
id : LongType()
userId : LongType()
total : DoubleType()
discountedTotal : DoubleType()

silver_posts
id : LongType()
userId : LongType()
views : LongType()


In [25]:
#Gold Customer Summary
#We'll use silver_users and silver_carts and join them using id = userId

from pyspark.sql import functions as F

users = spark.table("dbo.silver_users")
carts = spark.table("dbo.silver_carts")

gold_customer_summary = (
    users.alias("u")
    .join(
        carts.alias("c"),
        F.col("u.id") == F.col("c.userId"),
        "left"
    )
    .groupBy(
        F.col("u.id"),
        F.col("u.firstName"),
        F.col("u.lastName"),
        F.col("u.email")
    )
    .agg(
        F.count("c.id").alias("total_carts"),
        F.sum("c.total").alias("total_cart_value"),
        F.sum("c.discountedTotal").alias("total_discounted_value")
    )
)

gold_customer_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer_summary")

display(gold_customer_summary.orderBy(
    F.col("total_cart_value").desc()
).limit(10))

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, de6d51de-80b7-43f4-97f9-f997a505fc62)

In [26]:
# Gold products performance 
from pyspark.sql import functions as F

products = spark.table("dbo.silver_products")

gold_product_performance = (
    products
    .groupBy("category")
    .agg(
        F.count("id").alias("total_products"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.sum("stock").alias("total_stock")
    )
)

gold_product_performance.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_product_performance")

display(
    gold_product_performance.orderBy(
        F.col("total_products").desc()
    )
)

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c60b3d4-55ef-49c1-b2f8-82767e2e7b26)

In [27]:
# Gold cart analysis 
from pyspark.sql import functions as F

carts = spark.table("dbo.silver_carts")

gold_cart_analysis = carts.agg(
    F.count("id").alias("total_carts"),
    F.round(F.sum("total"), 2).alias("total_cart_value"),
    F.round(F.sum("discountedTotal"), 2).alias("total_discounted_value"),
    F.round(F.avg("total"), 2).alias("average_cart_value"),
    F.round(
        F.avg(F.col("total") - F.col("discountedTotal")), 2
    ).alias("average_discount_amount")
)

gold_cart_analysis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_cart_analysis")

display(gold_cart_analysis)

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2024ac35-4332-4aec-9bdd-cf23766a7a8b)

In [30]:
# Gold post engagement 
from pyspark.sql import functions as F

posts = spark.table("dbo.silver_posts")

gold_post_engagement = (
    posts
    .groupBy("userId")
    .agg(
        F.count("id").alias("total_posts"),
        F.sum("views").alias("total_views"),
        F.round(F.avg("views"), 2).alias("average_views")
    )
)

gold_post_engagement.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_post_engagement")

display(gold_post_engagement.orderBy(
    F.col("total_views").desc()
))

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d252240d-8921-4c76-9cdd-762ec3b3272b)

In [31]:
# validate all gold tables 
gold_tables = [
    "gold_customer_summary",
    "gold_product_performance",
    "gold_cart_analysis",
    "gold_post_engagement"
]

for table in gold_tables:
    df = spark.table(f"dbo.{table}")
    print(table, ":", df.count(), "records")

StatementMeta(, 44029f06-c5b8-437b-9af1-54baa9915a34, 33, Finished, Available, Finished, False)

gold_customer_summary : 30 records
gold_product_performance : 4 records
gold_cart_analysis : 1 records
gold_post_engagement : 29 records


In [1]:
from pyspark.sql import functions as F

users = spark.table("dbo.silver_users")

dim_customer = users.select(
    "id",
    "firstName",
    "lastName",
    "email"
).dropDuplicates(["id"])

dim_customer.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_customer")

print("Customer dimension:", dim_customer.count())

StatementMeta(, fcc31dc3-14af-4ba5-9289-7f4907c3b4d0, 3, Finished, Available, Finished, False)

Customer dimension: 30


In [2]:
products = spark.table("dbo.silver_products")

dim_category = products.select(
    "category"
).dropDuplicates()

dim_category.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_category")

print("Category dimension:", dim_category.count())

StatementMeta(, fcc31dc3-14af-4ba5-9289-7f4907c3b4d0, 4, Finished, Available, Finished, False)

Category dimension: 4
